# Context-Aware Multimodal Knowledge Retrieval System
## Complete End-to-End RAG Pipeline for PDF / Research Paper Analysis

**What this notebook does:**
- Extracts **all content** from any PDF: text, tables, images, charts, figures, equations
- Summarizes each modality using the best LLM for that content type:
  - **Groq / Llama-3.1** → fast, free text & table summarization
  - **Gemini 1.5 Flash (Vision)** → image, chart, figure, diagram understanding
- Embeds summaries using **HuggingFace sentence-transformers**
- Stores in **ChromaDB** with a **Multi-Vector Retrieval** strategy (summary → original)
- On any query: retrieves relevant text + tables + images and generates a cited answer via **Gemini**
- Shows all **source elements** inline: rendered tables (HTML), inline images, text with page refs

---
### Architecture
```
PDF ──► Unstructured Parser ──► Text / Tables / Images
            │                        │
            │         ┌──────────────┤──────────────────────┐
            │         ▼              ▼                       ▼
            │    Groq/Llama     Groq/Llama           Gemini Vision
            │    Text Summary   Table Summary         Image Summary
            │         │              │                       │
            │         └──────────────┴───────────────────────┘
            │                        │
            │              HuggingFace Embeddings
            │                        │
            │                   ChromaDB
            │           (summaries → original docs)
            │                        │
            └──────────► MultiVectorRetriever
                                     │
                              User Question
                                     │
                           Gemini 1.5 Flash
                                     │
                    ┌────────────────┴────────────────┐
                    ▼                                 ▼
             Answer (Cited)              Sources (Text/Table/Image)
```

## Step 1 — Install & Verify Dependencies
All packages are pre-installed in the `multimodal_env` virtual environment.
Run the cell below only if you need to install/update packages.

In [ ]:
# ── Optional: Install / update packages ────────────────────────────────────────
# Uncomment and run if any import fails below

# import subprocess, sys
# pkgs = [
#     "python-dotenv", "unstructured[all-docs]", "pdfplumber", "pymupdf", "pillow", "lxml",
#     "langchain>=0.3", "langchain-core>=0.3", "langchain-community>=0.3",
#     "langchain-google-genai>=2.0", "langchain-groq>=0.2", "langchain-huggingface>=0.1",
#     "google-generativeai>=0.8", "sentence-transformers>=3.0", "chromadb>=0.5",
#     "tiktoken", "pandas", "numpy", "opencv-python-headless", "matplotlib",
#     "ipywidgets", "tqdm",
# ]
# for pkg in pkgs:
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
# print("Done.")

# ── Verify key imports ─────────────────────────────────────────────────────────
import importlib, sys

required = {
    "dotenv":               "python-dotenv",
    "unstructured":         "unstructured[all-docs]",
    "pdfplumber":           "pdfplumber",
    "fitz":                 "pymupdf",
    "langchain":            "langchain",
    "langchain_community":  "langchain-community",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_groq":       "langchain-groq",
    "langchain_huggingface":"langchain-huggingface",
    "google.generativeai":  "google-generativeai",
    "sentence_transformers":"sentence-transformers",
    "chromadb":             "chromadb",
    "tiktoken":             "tiktoken",
    "cv2":                  "opencv-python-headless",
    "matplotlib":           "matplotlib",
    "ipywidgets":           "ipywidgets",
    "tqdm":                 "tqdm",
}

all_ok = True
for module, pkg in required.items():
    try:
        importlib.import_module(module)
        print(f"  ✓  {module}")
    except ImportError:
        print(f"  ✗  {module}  →  pip install {pkg}")
        all_ok = False

print()
print("All imports OK ✓" if all_ok else "⚠  Some packages missing — run the install block above.")
print(f"Python: {sys.version}")

## Step 2 — Environment Setup (API Keys)
API keys are loaded from `.env` in the project root. **Never hard-code keys in code.**

In [ ]:
import os
from dotenv import load_dotenv

# Load keys from .env (project root)
load_dotenv(dotenv_path=".env", override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY",   "")
HF_TOKEN       = os.getenv("HF_TOKEN",        "")

# Propagate to environment so LangChain adapters pick them up
os.environ["GOOGLE_API_KEY"]            = GOOGLE_API_KEY
os.environ["GROQ_API_KEY"]             = GROQ_API_KEY
os.environ["HF_TOKEN"]                 = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN

def _check(name, val):
    status = "✓ loaded" if val else "✗ MISSING — add to .env"
    print(f"  {name:<30} {status}")

print("API Key Status:")
_check("GOOGLE_API_KEY (Gemini)",  GOOGLE_API_KEY)
_check("GROQ_API_KEY",             GROQ_API_KEY)
_check("HF_TOKEN (HuggingFace)",   HF_TOKEN)

print()
if not all([GOOGLE_API_KEY, GROQ_API_KEY, HF_TOKEN]):
    print("⚠  One or more keys missing. Edit .env and re-run this cell.")
else:
    print("All API keys loaded ✓")

In [ ]:
import os
from pathlib import Path

# ── Directory constants (used throughout notebook) ──────────────────────────
CONTENT_DIR = "./content"          # PDF storage
IMAGES_DIR  = "./content/images"   # extracted image files (optional)
CHROMA_DIR  = "./chroma_db"        # persistent ChromaDB storage

for d in [CONTENT_DIR, IMAGES_DIR, CHROMA_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"  Ready: {d}")

print("\nDirectories set up ✓")

## Step 3 — Load Your PDF
Place your PDF in the `content/` folder, or use the helper below to copy it from any path.
You can also run the **widget uploader** for drag-and-drop upload inside Jupyter.

In [ ]:
import shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ── Option A: copy PDF from any path on your machine ───────────────────────
def load_pdf_from_path(source_path: str, rename: str = "paper.pdf") -> str:
    """
    Copy a PDF to content/ for processing.
    Usage: PDF_PATH = load_pdf_from_path(r"C:\\Users\\You\\Downloads\\paper.pdf")
    """
    src = Path(source_path)
    if not src.exists():
        raise FileNotFoundError(f"File not found: {source_path}")
    dest = Path(CONTENT_DIR) / rename
    shutil.copy2(src, dest)
    print(f"PDF copied → {dest}")
    return str(dest)

# ── Option B: drag-and-drop uploader widget ────────────────────────────────
def pdf_uploader_widget():
    """
    Interactive widget: drag a PDF onto the upload button.
    Returns a dict {'path': ...} that updates when upload is done.
    """
    uploader = widgets.FileUpload(accept=".pdf", multiple=False,
                                   description="Upload PDF",
                                   button_style="primary")
    status   = widgets.Output()
    state    = {"path": None}

    def _on_upload(change):
        with status:
            clear_output()
            for fname, fdata in uploader.value.items():
                dest = Path(CONTENT_DIR) / fname
                dest.write_bytes(fdata["content"])
                state["path"] = str(dest)
                print(f"✓ Uploaded: {dest}  ({len(fdata['content'])//1024} KB)")

    uploader.observe(_on_upload, names="value")
    display(widgets.VBox([
        widgets.HTML("<b>Upload a PDF to analyse:</b>"),
        uploader, status
    ]))
    return state   # access state["path"] after uploading

# ── Set PDF_PATH ────────────────────────────────────────────────────────────
#  → Option A (path copy):
#     PDF_PATH = load_pdf_from_path(r"C:\path\to\your.pdf")
#
#  → Option B (widget upload):
#     upload_state = pdf_uploader_widget()
#     # after uploading: PDF_PATH = upload_state["path"]
#
#  → Option C (already in content/):
PDF_PATH = "./content/paper.pdf"

print(f"PDF_PATH set to: {PDF_PATH}")
if not Path(PDF_PATH).exists():
    print()
    print("⚠  File not found at that path yet.")
    print("   Use load_pdf_from_path() or the upload widget, OR copy your PDF to ./content/paper.pdf")
else:
    size_kb = Path(PDF_PATH).stat().st_size // 1024
    print(f"   File exists ✓  ({size_kb} KB)")

## Step 4 — Extract All Multimodal Elements from PDF
Uses **Unstructured** with `hi_res` strategy (powered by a document layout model) to detect and extract:
- Text blocks (paragraphs, headers, captions)
- Tables (with HTML representation)
- Images / Figures / Charts / Diagrams (as base64)
- Equations and form elements are extracted as text where possible

> **Note:** `hi_res` strategy is thorough but slow. For a 20-page paper: ~1-3 min.

In [ ]:
from unstructured.partition.pdf import partition_pdf
from pathlib import Path

def extract_pdf_elements(pdf_path: str) -> list:
    """
    Partition a PDF into multimodal elements using Unstructured hi_res.

    Returns a list of chunks (CompositeElement, Table, etc.).
    Each CompositeElement may contain embedded Image elements in
    chunk.metadata.orig_elements.
    """
    if not Path(pdf_path).exists():
        raise FileNotFoundError(
            f"PDF not found: {pdf_path}\n"
            "Set PDF_PATH correctly in Step 3 and re-run."
        )

    print(f"Partitioning: {pdf_path}")
    print("Strategy: hi_res  (layout model + OCR)  — this may take a few minutes …\n")

    chunks = partition_pdf(
        filename=pdf_path,
        infer_table_structure=True,            # extract table structure as HTML
        strategy="hi_res",                     # best quality; requires poppler + tesseract
        extract_image_block_types=["Image"],   # pull figures/charts as base64
        extract_image_block_to_payload=True,   # embed base64 in metadata (no disk writes needed)
        chunking_strategy="by_title",          # group paragraphs under their section heading
        max_characters=10000,
        combine_text_under_n_chars=2000,
        new_after_n_chars=6000,
    )

    types = {}
    for c in chunks:
        t = type(c).__name__
        types[t] = types.get(t, 0) + 1

    print(f"Extraction complete. {len(chunks)} chunks found:")
    for t, n in sorted(types.items()):
        print(f"  {t}: {n}")

    return chunks

# ── Run extraction ──────────────────────────────────────────────────────────
chunks = extract_pdf_elements(PDF_PATH)

In [ ]:
def categorize_elements(chunks: list) -> dict:
    """
    Separate chunks into texts, tables, and images (base64 strings).

    - texts  : CompositeElement objects (paragraphs + embedded structure)
    - tables : Table objects (have .metadata.text_as_html)
    - images : list of base64 strings extracted from CompositeElement orig_elements
    """
    texts  = []
    tables = []
    images = []

    for chunk in chunks:
        ctype = type(chunk).__name__

        if "Table" in ctype:
            tables.append(chunk)

        elif "CompositeElement" in ctype:
            texts.append(chunk)

            # harvest any images embedded inside this composite chunk
            orig = getattr(getattr(chunk, "metadata", None), "orig_elements", None)
            if orig:
                for el in orig:
                    if "Image" in type(el).__name__:
                        b64 = getattr(getattr(el, "metadata", None), "image_base64", None)
                        if b64:
                            images.append(b64)

    print("Element breakdown:")
    print(f"  Text chunks : {len(texts)}")
    print(f"  Tables      : {len(tables)}")
    print(f"  Images      : {len(images)}")
    return {"texts": texts, "tables": tables, "images": images}

elements = categorize_elements(chunks)
texts    = elements["texts"]
tables   = elements["tables"]
images   = elements["images"]

## Step 5 — Inspect Extracted Elements
Preview each modality to confirm extraction quality before summarization.

In [ ]:
import base64
from IPython.display import display, HTML, Image as IPImage

# ────────────────────────────────────────────────────────────────────────────
# Shared display helpers (used throughout the notebook)
# ────────────────────────────────────────────────────────────────────────────

def display_image_b64(b64_str: str, caption: str = "", max_width: int = 700):
    """Render a base64-encoded image inline in Jupyter."""
    display(HTML(f"""
    <div style="margin:12px 0; text-align:center;">
      <img src="data:image/jpeg;base64,{b64_str}"
           style="max-width:{max_width}px; border:1px solid #ccc; border-radius:6px; box-shadow:0 2px 6px rgba(0,0,0,.15);"/>
      <p style="font-style:italic; color:#555; margin:4px 0;">{caption}</p>
    </div>
    """))

def display_table_chunk(table_chunk, index: int = 0):
    """Render a Table element as an HTML table in Jupyter."""
    page = getattr(getattr(table_chunk, "metadata", None), "page_number", "?")
    html = getattr(getattr(table_chunk, "metadata", None), "text_as_html", "")
    if not html:
        html = f"<pre>{table_chunk.text[:500]}</pre>"
    display(HTML(f"""
    <details open>
      <summary style="font-weight:bold; cursor:pointer;">
        Table {index + 1} &nbsp;|&nbsp; Page {page}
      </summary>
      <div style="overflow-x:auto; margin:8px 0; padding:4px;">
        {html}
      </div>
    </details>
    """))

def display_text_chunk(text_chunk, index: int = 0, max_chars: int = 600):
    """Render a text chunk as a styled card."""
    page = getattr(getattr(text_chunk, "metadata", None), "page_number", "?")
    preview = text_chunk.text[:max_chars].replace("<", "&lt;").replace(">", "&gt;")
    if len(text_chunk.text) > max_chars:
        preview += " …"
    display(HTML(f"""
    <details>
      <summary style="font-weight:bold; cursor:pointer;">
        Text Chunk {index + 1} &nbsp;|&nbsp; Page {page}
      </summary>
      <pre style="background:#f7f7f7; padding:10px; border-left:3px solid #007acc;
                  white-space:pre-wrap; font-size:13px;">{preview}</pre>
    </details>
    """))

print("Display helpers defined ✓")

In [ ]:
# ── Preview images ──────────────────────────────────────────────────────────
MAX_PREVIEW = 6

print(f"Showing up to {MAX_PREVIEW} of {len(images)} extracted images:\n")
if images:
    for i, img_b64 in enumerate(images[:MAX_PREVIEW]):
        display_image_b64(img_b64, caption=f"Figure {i + 1}")